# Gold Hiring Trends Mart

**Purpose**: Aggregated hiring trends by date and sector for time-series analysis

**Target Table**: `workspace.gold.gold_hiring_trends`

**Grain**: Date × Sector

**Refresh Strategy**: Full refresh (CREATE OR REPLACE) with metadata logging

**Parameters**:
- `lookback_days`: Number of days of historical data (default: 365)
- `force_full_refresh`: Boolean flag to force table recreation

**Key Metrics**:
- `total_new_jobs`: New jobs posted
- `total_active_jobs`: Active job count
- `total_closed_jobs`: Closed job count
- `unique_companies`: Companies hiring
- `avg_days_to_fill`: Average time to fill
- `updated_at`: Last refresh timestamp

**Data Sources**:
- `workspace.warehouse.fact_job_postings`
- `workspace.warehouse.dim_sector`

**Schema** (per contract):
- Primary key: (hiring_date_sk, sector_sk)

**Metadata Logging**:
- Tracks run history in `workspace.metadata.gold_hiring_trends_refresh_log`
- Captures: rows processed, unique dates/sectors, processing time, status

In [0]:
# Configuration
target_table = "workspace.gold.gold_hiring_trends"
metadata_table = "workspace.metadata.gold_hiring_trends_refresh_log"

# Parameters
lookback_days = 365  # How many days of history to include
force_full_refresh = False  # Set to True to drop and recreate table

In [0]:
import uuid
from datetime import datetime

# Generate unique run ID
run_id = str(uuid.uuid4())
run_timestamp = datetime.now()
status = "RUNNING"
error_message = None

print(f"Run ID: {run_id}")
print(f"Run Timestamp: {run_timestamp}")
print(f"Lookback Days: {lookback_days}")
print(f"Force Full Refresh: {force_full_refresh}")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.metadata.gold_hiring_trends_refresh_log (
  run_id STRING NOT NULL,
  run_timestamp TIMESTAMP NOT NULL,
  rows_inserted BIGINT,
  status STRING NOT NULL,
  lookback_days INT,
  rows_processed BIGINT,
  unique_dates INT,
  unique_sectors INT,
  force_full_refresh BOOLEAN,
  processing_time_seconds DECIMAL(10,2),
  error_message STRING
)
USING DELTA
COMMENT 'Audit log for gold_hiring_trends refresh operations';

In [0]:
# Optionally drop table for full refresh
if force_full_refresh:
    print(f"Force full refresh enabled - dropping table {target_table}")
    spark.sql(f"DROP TABLE IF EXISTS {target_table}")
    print("✓ Table dropped")
else:
    print("Proceeding with CREATE OR REPLACE (preserves table properties)")

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_hiring_trends (
  hiring_date_sk BIGINT NOT NULL COMMENT 'Date key',
  sector_sk BIGINT NOT NULL COMMENT 'Sector key',
  total_new_jobs BIGINT COMMENT 'New jobs posted',
  total_active_jobs BIGINT COMMENT 'Active job count',
  total_closed_jobs BIGINT COMMENT 'Closed job count',
  unique_companies BIGINT COMMENT 'Companies hiring',
  avg_days_to_fill DECIMAL(10,2) COMMENT 'Average time to fill',
  updated_at TIMESTAMP NOT NULL COMMENT 'Last refresh'
)
USING DELTA
COMMENT 'Aggregated hiring trends by date and sector for time-series analysis'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
%sql
INSERT OVERWRITE TABLE workspace.gold.gold_hiring_trends

WITH daily_metrics AS (
  SELECT
    f.posting_date_sk AS hiring_date_sk,
    f.sector_sk,
    COUNT(CASE WHEN f.is_new_job = TRUE THEN 1 END) AS total_new_jobs,
    COUNT(CASE WHEN f.active_flag = TRUE THEN 1 END) AS total_active_jobs,
    COUNT(CASE WHEN f.is_soft_delete = TRUE THEN 1 END) AS total_closed_jobs,
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.company_sk END) AS unique_companies,
    AVG(CASE 
      WHEN f.is_soft_delete = TRUE THEN 
        DATEDIFF(
          TO_DATE(CAST(f.posting_date_sk AS STRING), 'yyyyMMdd'),
          f.posting_timestamp
        )
      ELSE NULL
    END) AS avg_days_to_fill
  FROM workspace.warehouse.fact_job_postings f
  WHERE f.posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
  GROUP BY f.posting_date_sk, f.sector_sk
)

SELECT
  hiring_date_sk,
  sector_sk,
  total_new_jobs,
  total_active_jobs,
  total_closed_jobs,
  unique_companies,
  CAST(avg_days_to_fill AS DECIMAL(10,2)) AS avg_days_to_fill,
  CURRENT_TIMESTAMP() AS updated_at
FROM daily_metrics
ORDER BY hiring_date_sk DESC, sector_sk;

In [0]:
import time

start_time = time.time()

try:
    # Get summary metrics from the gold table
    metrics = spark.sql(f"""
        SELECT 
            COUNT(*) as rows_processed,
            COUNT(DISTINCT hiring_date_sk) as unique_dates,
            COUNT(DISTINCT sector_sk) as unique_sectors
        FROM {target_table}
    """).collect()[0]
    
    rows_processed = metrics['rows_processed']
    unique_dates = metrics['unique_dates']
    unique_sectors = metrics['unique_sectors']
    processing_time = time.time() - start_time
    status = "SUCCESS"
    
    print(f"✓ Processed {rows_processed:,} rows")
    print(f"✓ Unique dates: {unique_dates}")
    print(f"✓ Unique sectors: {unique_sectors}")
    print(f"✓ Processing time: {processing_time:.2f}s")
    
except Exception as e:
    status = "FAILED"
    error_message = str(e)
    processing_time = time.time() - start_time
    rows_processed = 0
    unique_dates = 0
    unique_sectors = 0
    print(f"✗ Error: {error_message}")
    raise

finally:
    # Write metadata log
    spark.sql(f"""
        INSERT INTO {metadata_table} (
            run_id,
            run_timestamp,
            rows_inserted,
            status,
            lookback_days,
            rows_processed,
            unique_dates,
            unique_sectors,
            force_full_refresh,
            processing_time_seconds,
            error_message
        )
        VALUES (
            '{run_id}',
            TIMESTAMP'{run_timestamp}',
            {rows_processed},
            '{status}',
            {lookback_days},
            {rows_processed},
            {unique_dates},
            {unique_sectors},
            {force_full_refresh},
            {processing_time:.2f},
            {'NULL' if error_message is None else f"'{error_message}'"}
        )
    """)

In [0]:
%sql
-- Summary statistics
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT hiring_date_sk) AS unique_dates,
  COUNT(DISTINCT sector_sk) AS unique_sectors,
  MIN(hiring_date_sk) AS earliest_date,
  MAX(hiring_date_sk) AS latest_date,
  SUM(total_new_jobs) AS total_new_jobs,
  SUM(total_active_jobs) AS total_active_jobs,
  SUM(total_closed_jobs) AS total_closed_jobs,
  AVG(avg_days_to_fill) AS avg_days_to_fill,
  MAX(updated_at) AS last_refreshed
FROM workspace.gold.gold_hiring_trends;

In [0]:
%sql
-- Check for data quality issues
SELECT
  'Null hiring_date_sk' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_hiring_trends
WHERE hiring_date_sk IS NULL

UNION ALL

SELECT
  'Null sector_sk' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_hiring_trends
WHERE sector_sk IS NULL

UNION ALL

SELECT
  'Negative job counts' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_hiring_trends
WHERE total_new_jobs < 0 
   OR total_active_jobs < 0 
   OR total_closed_jobs < 0

UNION ALL

SELECT
  'Invalid avg_days_to_fill' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_hiring_trends
WHERE avg_days_to_fill < 0 OR avg_days_to_fill > 1000

ORDER BY issue_count DESC;

In [0]:
%sql
-- Top 10 sectors by total new jobs
SELECT
  sector_sk,
  SUM(total_new_jobs) AS total_new_jobs,
  SUM(total_active_jobs) AS total_active_jobs,
  AVG(avg_days_to_fill) AS avg_days_to_fill,
  COUNT(DISTINCT hiring_date_sk) AS days_active
FROM workspace.gold.gold_hiring_trends
GROUP BY sector_sk
ORDER BY total_new_jobs DESC
LIMIT 10;

In [0]:
%sql
-- Last 30 days trend
SELECT
  hiring_date_sk,
  COUNT(DISTINCT sector_sk) AS active_sectors,
  SUM(total_new_jobs) AS daily_new_jobs,
  SUM(total_active_jobs) AS daily_active_jobs,
  AVG(avg_days_to_fill) AS avg_days_to_fill
FROM workspace.gold.gold_hiring_trends
WHERE hiring_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 30), 'yyyyMMdd') AS INT)
GROUP BY hiring_date_sk
ORDER BY hiring_date_sk DESC;

In [0]:
%sql
-- View last 10 refresh runs
SELECT 
  run_id,
  run_timestamp,
  lookback_days,
  rows_processed,
  unique_dates,
  unique_sectors,
  processing_time_seconds,
  status,
  error_message
FROM workspace.metadata.gold_hiring_trends_refresh_log
ORDER BY run_timestamp DESC
LIMIT 10;